In [1]:
#!/usr/bin/env python3
"""
NeuroPipe extension: TTFS encoding + Adaptive-LIF SNN.
Uses the public NEXUS processed dataset format. Does not modify the original NEXUS code.

Expected CSV columns in current repo:
  Flow_L_min, label, TDS_smooth, Turbidity_smooth
If all six features are available, set FEATURE_COLUMNS accordingly. The checked-in
CSV inspected for this implementation contains only these 3 input features plus label.
This script intentionally does not invent missing time-derived features.
"""
from __future__ import annotations

import argparse
import json
import random
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from torch import nn
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import (
    accuracy_score, classification_report, confusion_matrix,
    f1_score, precision_score, recall_score
)
import matplotlib.pyplot as plt

SEED = 42
FEATURE_COLUMNS = ["Flow_L_min", "TDS_smooth", "Turbidity_smooth"]
TARGET_COLUMN = "label"
CLASS_NAMES = {
    0: "Normal clean-water flow",
    1: "Contaminated flow",
    2: "Muddy water / partial blockage",
    3: "Severe blockage",
}

def seed_everything(seed: int = SEED) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

class TTFSAdaptiveLIF(nn.Module):
    """Small feed-forward spiking classifier trained with surrogate gradients.

    TTFS: each normalized feature emits at most one spike; larger values emit
    earlier in the T-step window. Adaptive-LIF adds a spike-history-dependent
    threshold term. This is a software simulation, not a neuromorphic chip.
    """
    def __init__(self, input_dim: int, hidden_dim: int, n_classes: int,
                 timesteps: int = 20, beta: float = 0.90,
                 adaptation: float = 0.25, threshold: float = 1.0):
        super().__init__()
        self.fc1 = nn.Linear(input_dim, hidden_dim)
        self.fc2 = nn.Linear(hidden_dim, n_classes)
        self.timesteps = timesteps
        self.beta = beta
        self.adaptation = adaptation
        self.threshold = threshold

    @staticmethod
    def surrogate_spike(x: torch.Tensor) -> torch.Tensor:
        # Fast-sigmoid surrogate gradient in backward pass.
        hard = (x >= 0).to(x.dtype)
        soft = torch.sigmoid(5.0 * x)
        return hard + soft - soft.detach()

    def forward(self, x: torch.Tensor, return_spike_count: bool = False):
        # x is scaled to [0,1]. A single spike occurs earlier for larger values.
        # t_first = floor((1 - x) * (T-1)); no spike for exactly zero.
        x_clip = torch.clamp(x, 0.0, 1.0)
        first_t = torch.floor((1.0 - x_clip) * (self.timesteps - 1)).long()
        v1 = torch.zeros(x.size(0), self.fc1.out_features, device=x.device)
        a1 = torch.zeros_like(v1)
        v2 = torch.zeros(x.size(0), self.fc2.out_features, device=x.device)
        a2 = torch.zeros_like(v2)
        out_sum = torch.zeros_like(v2)
        spike_count = torch.zeros((), device=x.device)

        for t in range(self.timesteps):
            input_spikes = ((first_t == t) & (x_clip > 0)).to(x.dtype)
            current1 = self.fc1(input_spikes)
            v1 = self.beta * v1 + current1 - a1
            s1 = self.surrogate_spike(v1 - (self.threshold + self.adaptation * a1))
            v1 = v1 * (1.0 - s1.detach())
            a1 = 0.9 * a1 + s1.detach()

            current2 = self.fc2(s1)
            v2 = self.beta * v2 + current2 - a2
            s2 = self.surrogate_spike(v2 - (self.threshold + self.adaptation * a2))
            v2 = v2 * (1.0 - s2.detach())
            a2 = 0.9 * a2 + s2.detach()
            out_sum = out_sum + s2
            spike_count = spike_count + s1.detach().sum() + s2.detach().sum()

        if return_spike_count:
            return out_sum, spike_count
        return out_sum

def main():
    parser = argparse.ArgumentParser()
    parser.add_argument("--data", type=Path, default=Path("Training/final_data.csv"))
    parser.add_argument("--outdir", type=Path, default=Path("results_neuropipe_adaptive"))
    parser.add_argument("--epochs", type=int, default=100)
    parser.add_argument("--batch-size", type=int, default=64)
    parser.add_argument("--hidden", type=int, default=32)
    parser.add_argument("--timesteps", type=int, default=20)
    parser.add_argument("--lr", type=float, default=1e-3)
    args = parser.parse_args()

    seed_everything()
    args.outdir.mkdir(parents=True, exist_ok=True)
    df = pd.read_csv(args.data)
    required = FEATURE_COLUMNS + [TARGET_COLUMN]
    missing = [c for c in required if c not in df.columns]
    if missing:
        raise ValueError(f"Missing columns {missing}. Found columns: {list(df.columns)}")

    df = df[required].replace([np.inf, -np.inf], np.nan).dropna()
    X = df[FEATURE_COLUMNS].to_numpy(dtype=np.float32)
    y = df[TARGET_COLUMN].to_numpy(dtype=np.int64)
    classes = np.unique(y)
    if not np.array_equal(classes, np.array([0, 1, 2, 3])):
        raise ValueError(f"Expected labels 0,1,2,3; found {classes.tolist()}")

    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.20, random_state=SEED, stratify=y
    )
    # Split train into train/validation; fit scaling on training subset only.
    X_train, X_val, y_train, y_val = train_test_split(
        X_train, y_train, test_size=0.20, random_state=SEED, stratify=y_train
    )
    # TTFS expects [0,1]. Fit MinMaxScaler on training subset only.
    scaler = MinMaxScaler()
    X_train = scaler.fit_transform(X_train)
    X_val = np.clip(scaler.transform(X_val), 0.0, 1.0)
    X_test = np.clip(scaler.transform(X_test), 0.0, 1.0)
    lo = scaler.data_min_
    hi = scaler.data_max_

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model = TTFSAdaptiveLIF(
        input_dim=len(FEATURE_COLUMNS), hidden_dim=args.hidden, n_classes=4,
        timesteps=args.timesteps
    ).to(device)
    # Class weights calculated from training split only.
    counts = np.bincount(y_train, minlength=4)
    weights = len(y_train) / (4.0 * np.maximum(counts, 1))
    criterion = nn.CrossEntropyLoss(weight=torch.tensor(weights, dtype=torch.float32, device=device))
    optimizer = torch.optim.AdamW(model.parameters(), lr=args.lr, weight_decay=1e-4)
    Xtr = torch.tensor(X_train, dtype=torch.float32)
    ytr = torch.tensor(y_train, dtype=torch.long)
    Xv = torch.tensor(X_val, dtype=torch.float32, device=device)
    yv = torch.tensor(y_val, dtype=torch.long, device=device)

    best_val_loss = float("inf")
    best_state = None
    patience, wait = 12, 0
    history = []
    for epoch in range(1, args.epochs + 1):
        model.train()
        order = torch.randperm(len(Xtr))
        total_loss = 0.0
        for idx in order.split(args.batch_size):
            xb = Xtr[idx].to(device)
            yb = ytr[idx].to(device)
            optimizer.zero_grad(set_to_none=True)
            logits = model(xb)
            loss = criterion(logits, yb)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            total_loss += loss.item() * len(idx)
        model.eval()
        with torch.no_grad():
            val_logits = model(Xv)
            val_loss = criterion(val_logits, yv).item()
            val_acc = accuracy_score(y_val, val_logits.argmax(1).cpu().numpy())
        row = {"epoch": epoch, "train_loss": total_loss / len(Xtr),
               "val_loss": val_loss, "val_accuracy": float(val_acc)}
        history.append(row)
        print(f"Epoch {epoch:03d}/{args.epochs} | train_loss={row['train_loss']:.4f} "
              f"| val_loss={val_loss:.4f} | val_acc={val_acc:.4f}")
        if val_loss < best_val_loss - 1e-5:
            best_val_loss = val_loss
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            wait = 0
        else:
            wait += 1
            if wait >= patience:
                print(f"Early stopping at epoch {epoch}.")
                break

    if best_state is not None:
        model.load_state_dict(best_state)
    model.eval()
    Xt = torch.tensor(X_test, dtype=torch.float32, device=device)
    with torch.no_grad():
        pred_scores, spike_count = model(Xt, return_spike_count=True)
        y_pred = pred_scores.argmax(1).cpu().numpy()
    metrics = {
        "n_samples_after_cleaning": int(len(df)),
        "n_train": int(len(y_train)), "n_validation": int(len(y_val)),
        "n_test": int(len(y_test)), "features": FEATURE_COLUMNS,
        "encoding": "TTFS, at most one input spike per feature per sample",
        "neuron": "Adaptive LIF with spike-history threshold adaptation",
        "timesteps": args.timesteps, "hidden_units": args.hidden,
        "accuracy": float(accuracy_score(y_test, y_pred)),
        "macro_f1": float(f1_score(y_test, y_pred, average="macro")),
        "weighted_f1": float(f1_score(y_test, y_pred, average="weighted")),
        "macro_precision": float(precision_score(y_test, y_pred, average="macro", zero_division=0)),
        "macro_recall": float(recall_score(y_test, y_pred, average="macro", zero_division=0)),
        "test_mean_spikes_per_sample": float(spike_count.item() / len(y_test)),
        "device": str(device),
    }
    print("\nTEST METRICS")
    print(json.dumps(metrics, indent=2))
    print("\nCLASSIFICATION REPORT")
    print(classification_report(y_test, y_pred, labels=[0,1,2,3],
          target_names=[CLASS_NAMES[i] for i in range(4)], zero_division=0))
    print("CONFUSION MATRIX (labels 0-3)")
    print(confusion_matrix(y_test, y_pred, labels=[0,1,2,3]))

    torch.save({"state_dict": model.state_dict(), "features": FEATURE_COLUMNS,
                "scaler_data_min": lo.tolist(), "scaler_data_max": hi.tolist(),
                "scaler_scale": scaler.scale_.tolist(), "scaler_min": scaler.min_.tolist(),
                "class_names": CLASS_NAMES, "timesteps": args.timesteps},
               args.outdir / "ttfs_adaptive_lif.pt")
    pd.DataFrame(history).to_csv(args.outdir / "training_history.csv", index=False)
    (args.outdir / "metrics.json").write_text(json.dumps(metrics, indent=2))
    (args.outdir / "classification_report.txt").write_text(
        classification_report(y_test, y_pred, labels=[0,1,2,3],
          target_names=[CLASS_NAMES[i] for i in range(4)], zero_division=0)
    )
    cm = confusion_matrix(y_test, y_pred, labels=[0,1,2,3])
    fig, ax = plt.subplots(figsize=(6,5))
    im = ax.imshow(cm, cmap="Blues")
    ax.set_xticks(range(4), labels=["Normal","Contaminated","Partial block","Severe block"], rotation=25, ha="right")
    ax.set_yticks(range(4), labels=["Normal","Contaminated","Partial block","Severe block"])
    ax.set_xlabel("Predicted"); ax.set_ylabel("True"); ax.set_title("TTFS Adaptive-LIF SNN")
    for (i,j), value in np.ndenumerate(cm):
        ax.text(j, i, str(value), ha="center", va="center")
    fig.colorbar(im, ax=ax); fig.tight_layout()
    fig.savefig(args.outdir / "confusion_matrix.png", dpi=180); plt.close(fig)
    print(f"\nArtifacts written to: {args.outdir.resolve()}")

if __name__ == "__main__":
    main()


usage: ipykernel_launcher.py [-h] [--data DATA] [--outdir OUTDIR]
                             [--epochs EPOCHS] [--batch-size BATCH_SIZE]
                             [--hidden HIDDEN] [--timesteps TIMESTEPS]
                             [--lr LR]
ipykernel_launcher.py: error: unrecognized arguments: --f=c:\Users\abhia\AppData\Roaming\jupyter\runtime\kernel-v3bb0298d1033b8950a5943df1695afce8a50db1fb.json


SystemExit: 2

c:\Users\abhia\anaconda3\Lib\site-packages\IPython\core\interactiveshell.py:3585: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)
